# 02 — Data Cleaning and Data Quality Preparation
## Financial Fraud Detection Project

---

### Purpose

This notebook is responsible for transforming the raw synthetic fraud dataset into a **clean, reliable dataset** that subsequent notebooks can safely use.

**What this notebook does:**
- Validates the raw dataset structure and schema
- Corrects data types where necessary
- Parses the `Date` column from text to `datetime`
- Checks for missing values
- Checks for duplicate rows
- Validates categorical column values
- Validates numerical columns for invalid entries
- Validates the target variable (`Fraud_Label`)
- Documents identifier column handling decisions
- Saves a cleaned dataset to `data/processed/cleaned_fraud_dataset.csv`

### Out of Scope

This notebook **intentionally does NOT** perform:
- Exploratory Data Analysis (EDA)
- Feature engineering or extraction
- Encoding (label encoding, one-hot encoding, etc.)
- Scaling or normalisation
- SMOTE or any class rebalancing
- Train/test splitting
- Model training or evaluation
- Dashboard development

### Input
`data/raw/synthetic_fraud_dataset.csv`

### Output
`data/processed/cleaned_fraud_dataset.csv`

---
## 2. Imports

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings

# Display options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)
pd.set_option("display.width", 120)

# Suppress non-critical warnings for cleaner output
warnings.filterwarnings("ignore", category=FutureWarning)

print("Libraries imported successfully.")

Libraries imported successfully.


---
## 3. Robust Project Root / Path Setup

In [2]:
def find_project_root(marker_files=("requirements.txt", "README.md", ".gitignore"), max_levels=5):
    """
    Walk upward from the current working directory to locate the project root.
    The root is identified by the presence of one of the marker files.
    """
    current = Path.cwd().resolve()
    for _ in range(max_levels):
        if any((current / m).exists() for m in marker_files):
            return current
        parent = current.parent
        if parent == current:
            break
        current = parent
    raise FileNotFoundError(
        "Could not locate project root. Ensure you run this notebook "
        "from within the project directory tree."
    )


PROJECT_ROOT = find_project_root()
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "synthetic_fraud_dataset.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_fraud_dataset.csv"

# Create processed directory if it does not exist
PROCESSED_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset not found at: {RAW_DATA_PATH}\n"
        "Please place 'synthetic_fraud_dataset.csv' inside data/raw/."
    )

print(f"Project root       : {PROJECT_ROOT}")
print(f"Raw data path      : {RAW_DATA_PATH}")
print(f"Processed data path: {PROCESSED_DATA_PATH}")

Project root       : C:\Users\sarva\Desktop\financial-fraud-detection
Raw data path      : C:\Users\sarva\Desktop\financial-fraud-detection\data\raw\synthetic_fraud_dataset.csv
Processed data path: C:\Users\sarva\Desktop\financial-fraud-detection\data\processed\cleaned_fraud_dataset.csv


---
## 4. Load Raw Dataset

In [3]:
# Load the raw dataset — this remains untouched throughout the notebook
df_raw = pd.read_csv(RAW_DATA_PATH)

print(f"Raw dataset shape: {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"\nColumn names: {list(df_raw.columns)}")

Raw dataset shape: 50,000 rows × 14 columns

Column names: ['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Fraud_Label']


In [4]:
df_raw.head()

,Transaction_ID,User_ID,Transaction_Amount,Transaction_Type,Date,Account_Balance,Device_Type,Location,Merchant_Category,Previous_Fraudulent_Activity,Daily_Transaction_Count,Card_Type,Card_Age,Fraud_Label
0,TXN_33553,USER_1834,39.79,POS,14 August 2023,93213.17,Laptop,Sydney,Travel,0,7,Amex,65,0
1,TXN_9427,USER_7875,1.19,Bank Transfer,7 June 2023,75725.25,Mobile,New York,Clothing,0,13,Mastercard,186,1
2,TXN_199,USER_2734,28.96,Online,20 June 2023,1588.96,Tablet,Mumbai,Restaurants,0,14,Visa,226,1
3,TXN_12447,USER_2617,254.32,ATM Withdrawal,7 December 2023,76807.20,Tablet,New York,Clothing,0,8,Visa,76,1
4,TXN_39489,USER_2014,31.28,POS,11 November 2023,92354.66,Mobile,Mumbai,Electronics,1,14,Mastercard,140,1


In [5]:
# Create a working copy — all cleaning operations happen on df_clean
# The raw dataset (df_raw) is never modified
df_clean = df_raw.copy()

print(f"Working copy created: {df_clean.shape[0]:,} rows × {df_clean.shape[1]} columns")

Working copy created: 50,000 rows × 14 columns


---
## 5. Initial Validation

Verify that all expected columns are present and no unexpected structural changes have occurred.

In [6]:
EXPECTED_COLUMNS = [
    "Transaction_ID",
    "User_ID",
    "Transaction_Amount",
    "Transaction_Type",
    "Date",
    "Account_Balance",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Previous_Fraudulent_Activity",
    "Daily_Transaction_Count",
    "Card_Type",
    "Card_Age",
    "Fraud_Label",
]

actual_columns = set(df_clean.columns)
expected_set = set(EXPECTED_COLUMNS)

missing_columns = expected_set - actual_columns
unexpected_columns = actual_columns - expected_set

if missing_columns:
    raise ValueError(
        f"Missing required columns: {sorted(missing_columns)}\n"
        f"Available columns: {sorted(actual_columns)}"
    )

print(f"✅ All {len(EXPECTED_COLUMNS)} expected columns are present.")
print(f"   Rows   : {df_clean.shape[0]:,}")
print(f"   Columns: {df_clean.shape[1]}")

if unexpected_columns:
    print(f"\n⚠️  Unexpected additional columns found: {sorted(unexpected_columns)}")
else:
    print("   No unexpected columns.")

✅ All 14 expected columns are present.
   Rows   : 50,000
   Columns: 14
   No unexpected columns.


---
## 6. Data Type Validation

Inspect current pandas dtypes and convert where logically justified.

In [7]:
# Current dtypes before any conversion
print("Current data types:\n")
for col in df_clean.columns:
    print(f"  {col:35s} → {df_clean[col].dtype}")

Current data types:

  Transaction_ID                      → str
  User_ID                             → str
  Transaction_Amount                  → float64
  Transaction_Type                    → str
  Date                                → str
  Account_Balance                     → float64
  Device_Type                         → str
  Location                            → str
  Merchant_Category                   → str
  Previous_Fraudulent_Activity        → int64
  Daily_Transaction_Count             → int64
  Card_Type                           → str
  Card_Age                            → int64
  Fraud_Label                         → int64


In [8]:
# Expected logical types mapping for reference
EXPECTED_TYPES = {
    "Transaction_ID":                "object (identifier)",
    "User_ID":                       "object (identifier)",
    "Transaction_Amount":            "float64",
    "Transaction_Type":              "object (categorical)",
    "Date":                          "datetime64",
    "Account_Balance":               "float64",
    "Device_Type":                   "object (categorical)",
    "Location":                      "object (categorical)",
    "Merchant_Category":             "object (categorical)",
    "Previous_Fraudulent_Activity":  "int64 (binary)",
    "Daily_Transaction_Count":       "int64",
    "Card_Type":                     "object (categorical)",
    "Card_Age":                      "int64",
    "Fraud_Label":                   "int64 (binary)",
}

dtype_comparison = pd.DataFrame({
    "Current Dtype": df_clean.dtypes.astype(str),
    "Expected Logical Type": pd.Series(EXPECTED_TYPES),
})
dtype_comparison

,Current Dtype,Expected Logical Type
Transaction_ID,str,object (identifier)
User_ID,str,object (identifier)
Transaction_Amount,float64,float64
Transaction_Type,str,object (categorical)
Date,str,datetime64
Account_Balance,float64,float64
Device_Type,str,object (categorical)
Location,str,object (categorical)
Merchant_Category,str,object (categorical)
Previous_Fraudulent_Activity,int64,int64 (binary)


In [9]:
# --- Safe numeric conversions ---
# Ensure numeric columns are indeed numeric; report any values that fail conversion

numeric_columns = [
    "Transaction_Amount",
    "Account_Balance",
    "Previous_Fraudulent_Activity",
    "Daily_Transaction_Count",
    "Card_Age",
    "Fraud_Label",
]

conversion_issues = []

for col in numeric_columns:
    original_non_null = df_clean[col].notna().sum()
    converted = pd.to_numeric(df_clean[col], errors="coerce")
    new_non_null = converted.notna().sum()
    coerced_to_nan = original_non_null - new_non_null

    if coerced_to_nan > 0:
        conversion_issues.append({
            "Column": col,
            "Values Coerced to NaN": coerced_to_nan,
        })
        print(f"⚠️  {col}: {coerced_to_nan} non-numeric values were coerced to NaN during conversion.")
    
    df_clean[col] = converted

if not conversion_issues:
    print("✅ All numeric columns converted successfully with no values coerced to NaN.")
else:
    issues_df = pd.DataFrame(conversion_issues).set_index("Column")
    print("\nConversion issues summary:")
    display(issues_df)

# Ensure identifier columns remain as strings
df_clean["Transaction_ID"] = df_clean["Transaction_ID"].astype(str)
df_clean["User_ID"] = df_clean["User_ID"].astype(str)

print("\nIdentifier columns confirmed as string type.")

✅ All numeric columns converted successfully with no values coerced to NaN.

Identifier columns confirmed as string type.


---
## 7. Missing Value Check

Check for missing values across all columns before any cleaning operations.

In [10]:
missing_count = df_clean.isnull().sum()
missing_pct = (df_clean.isnull().sum() / len(df_clean)) * 100

missing_df = pd.DataFrame({
    "Column": df_clean.columns,
    "Missing Count": missing_count.values,
    "Missing Percentage (%)": missing_pct.values.round(4),
})

missing_df = missing_df.sort_values("Missing Count", ascending=False).reset_index(drop=True)
missing_df

,Column,Missing Count,Missing Percentage (%)
0,Transaction_ID,0,0.00
1,User_ID,0,0.00
2,Transaction_Amount,0,0.00
3,Transaction_Type,0,0.00
4,Date,0,0.00
5,Account_Balance,0,0.00
6,Device_Type,0,0.00
7,Location,0,0.00
8,Merchant_Category,0,0.00
9,Previous_Fraudulent_Activity,0,0.00


In [11]:
total_missing = missing_count.sum()

if total_missing == 0:
    print("✅ No missing values were found; therefore no imputation was required.")
else:
    cols_with_missing = missing_df[missing_df["Missing Count"] > 0]
    print(f"⚠️  {total_missing} total missing values found across {len(cols_with_missing)} column(s).")
    print("\nColumns with missing values:")
    display(cols_with_missing)
    print("\nNote: These may have been introduced during numeric type conversion (Section 6).")
    print("Investigate the specific rows before deciding on an imputation strategy.")

✅ No missing values were found; therefore no imputation was required.


---
## 8. Duplicate Check

Check for exact duplicate rows across all columns.

In [12]:
num_duplicates = df_clean.duplicated().sum()
dup_pct = (num_duplicates / len(df_clean)) * 100

print(f"Exact duplicate rows : {num_duplicates:,}")
print(f"Duplicate percentage : {dup_pct:.2f}%")

if num_duplicates > 0:
    print(f"\n⚠️  {num_duplicates:,} duplicate rows found. Displaying a sample:")
    display(df_clean[df_clean.duplicated(keep=False)].head(10))
    
    # Remove exact duplicates, keeping the first occurrence
    rows_before = len(df_clean)
    df_clean = df_clean.drop_duplicates(keep="first").reset_index(drop=True)
    rows_removed = rows_before - len(df_clean)
    print(f"\n   Removed {rows_removed:,} duplicate rows.")
    print(f"   Rows remaining: {len(df_clean):,}")
else:
    print("\n✅ No exact duplicate rows detected. No removal needed.")

Exact duplicate rows : 0
Duplicate percentage : 0.00%

✅ No exact duplicate rows detected. No removal needed.


---
## 9. Date Cleaning and Parsing

The raw `Date` column is stored as text (e.g., `"14 August 2023"`). Convert it to a proper `datetime` type.

> **Note:** This section only converts the data type. Date-based feature extraction (year, month, day, weekday, hour) is a **feature engineering** task for Notebook 04.

In [13]:
print(f"Date column dtype before conversion: {df_clean['Date'].dtype}")
print(f"Sample values:")
print(df_clean["Date"].head(10).tolist())

Date column dtype before conversion: str
Sample values:
['14 August 2023', '7 June 2023', '20 June 2023', '7 December 2023', '11 November 2023', '5 June 2023', '7 November 2023', '25 February 2023', '9 March 2023', '20 September 2023']


In [14]:
# Parse Date column with a specified format matching the data pattern: "14 August 2023"
df_clean["Date"] = pd.to_datetime(df_clean["Date"], format="%d %B %Y", errors="coerce")

# Report parsing results
n_parsed = df_clean["Date"].notna().sum()
n_failed = df_clean["Date"].isna().sum()

print(f"Date column dtype after conversion: {df_clean['Date'].dtype}")
print(f"\nSuccessfully parsed dates: {n_parsed:,}")
print(f"Unparseable dates (NaT) : {n_failed:,}")

if n_parsed > 0:
    print(f"\nMinimum date: {df_clean['Date'].min()}")
    print(f"Maximum date: {df_clean['Date'].max()}")
    print(f"Date range  : {df_clean['Date'].max() - df_clean['Date'].min()}")

if n_failed > 0:
    print(f"\n⚠️  {n_failed:,} dates could not be parsed. Displaying affected rows:")
    display(df_raw.loc[df_clean["Date"].isna(), ["Transaction_ID", "Date"]].head(20))
else:
    print("\n✅ All dates parsed successfully.")

Date column dtype after conversion: datetime64[us]

Successfully parsed dates: 50,000
Unparseable dates (NaT) : 0

Minimum date: 2023-01-01 00:00:00
Maximum date: 2023-12-31 00:00:00
Date range  : 364 days 00:00:00

✅ All dates parsed successfully.


---
## 10. Categorical Value Validation

Validate the known categorical columns for blank strings, missing values, and unexpected entries.

> **Note:** This section does **not** encode categories or create dummy variables. It only validates data quality.

In [15]:
CATEGORICAL_COLUMNS = [
    "Transaction_Type",
    "Device_Type",
    "Location",
    "Merchant_Category",
    "Card_Type",
]

for col in CATEGORICAL_COLUMNS:
    unique_vals = df_clean[col].unique()
    n_unique = len(unique_vals)
    n_missing = df_clean[col].isna().sum()
    
    # Check for blank/whitespace-only strings
    blank_mask = df_clean[col].astype(str).str.strip().eq("")
    n_blank = blank_mask.sum()
    
    print(f"\n{'=' * 50}")
    print(f"{col}")
    print(f"{'=' * 50}")
    print(f"  Unique values : {n_unique}")
    print(f"  Missing (NaN) : {n_missing}")
    print(f"  Blank strings : {n_blank}")
    print(f"  Values        : {sorted(df_clean[col].dropna().unique().tolist())}")
    
    if n_blank > 0:
        print(f"  ⚠️  {n_blank} blank string(s) detected — may need cleaning.")
    if n_missing > 0:
        print(f"  ⚠️  {n_missing} missing value(s) detected.")


Transaction_Type
  Unique values : 4
  Missing (NaN) : 0
  Blank strings : 0
  Values        : ['ATM Withdrawal', 'Bank Transfer', 'Online', 'POS']

Device_Type
  Unique values : 3
  Missing (NaN) : 0
  Blank strings : 0
  Values        : ['Laptop', 'Mobile', 'Tablet']

Location
  Unique values : 5
  Missing (NaN) : 0
  Blank strings : 0
  Values        : ['London', 'Mumbai', 'New York', 'Sydney', 'Tokyo']

Merchant_Category
  Unique values : 5
  Missing (NaN) : 0
  Blank strings : 0
  Values        : ['Clothing', 'Electronics', 'Groceries', 'Restaurants', 'Travel']

Card_Type
  Unique values : 4
  Missing (NaN) : 0
  Blank strings : 0
  Values        : ['Amex', 'Discover', 'Mastercard', 'Visa']


---
## 11. Numerical Value Validation

Check numerical columns for invalid values (e.g., unexpected negatives, impossible binary values, NaN introduced during conversion).

> **Note:** This section does **not** remove statistical outliers. Outlier analysis is a later analysis/feature-engineering concern.

In [16]:
NUMERICAL_VALIDATION = {
    "Transaction_Amount":  {"allow_negative": False, "binary": False},
    "Account_Balance":     {"allow_negative": True,  "binary": False},  # overdrafts possible
    "Previous_Fraudulent_Activity": {"allow_negative": False, "binary": True, "valid_values": {0, 1}},
    "Daily_Transaction_Count":     {"allow_negative": False, "binary": False},
    "Card_Age":            {"allow_negative": False, "binary": False},
}

for col, rules in NUMERICAL_VALIDATION.items():
    print(f"\n{'=' * 50}")
    print(f"{col}")
    print(f"{'=' * 50}")
    
    n_nan = df_clean[col].isna().sum()
    print(f"  NaN values       : {n_nan}")
    
    valid_data = df_clean[col].dropna()
    
    if len(valid_data) == 0:
        print("  ⚠️  Column is entirely NaN — investigate.")
        continue
    
    print(f"  Min              : {valid_data.min():.4f}")
    print(f"  Max              : {valid_data.max():.4f}")
    print(f"  Mean             : {valid_data.mean():.4f}")
    
    # Check for negative values where not allowed
    if not rules["allow_negative"]:
        n_negative = (valid_data < 0).sum()
        if n_negative > 0:
            print(f"  ⚠️  {n_negative} negative value(s) found (not expected for this column).")
        else:
            print(f"  ✅ No unexpected negative values.")
    else:
        n_negative = (valid_data < 0).sum()
        print(f"  Negative values  : {n_negative} (allowed for this column)")
    
    # Check binary column validity
    if rules.get("binary"):
        actual_values = set(valid_data.unique())
        expected_values = rules["valid_values"]
        invalid_values = actual_values - expected_values
        if invalid_values:
            print(f"  ⚠️  Invalid values for binary column: {invalid_values}")
        else:
            print(f"  ✅ Binary values valid: {sorted(actual_values)}")


Transaction_Amount
  NaN values       : 0
  Min              : 0.0000
  Max              : 1174.1400
  Mean             : 99.4110
  ✅ No unexpected negative values.

Account_Balance
  NaN values       : 0
  Min              : 500.4800
  Max              : 99998.3100
  Mean             : 50294.0660
  Negative values  : 0 (allowed for this column)

Previous_Fraudulent_Activity
  NaN values       : 0
  Min              : 0.0000
  Max              : 1.0000
  Mean             : 0.0984
  ✅ No unexpected negative values.
  ✅ Binary values valid: [np.int64(0), np.int64(1)]

Daily_Transaction_Count
  NaN values       : 0
  Min              : 1.0000
  Max              : 14.0000
  Mean             : 7.4852
  ✅ No unexpected negative values.

Card_Age
  NaN values       : 0
  Min              : 1.0000
  Max              : 239.0000
  Mean             : 119.9999
  ✅ No unexpected negative values.


---
## 12. Target Validation

Validate the `Fraud_Label` target variable.

In [17]:
TARGET_COL = "Fraud_Label"

print(f"Target column: {TARGET_COL}")
print(f"Dtype        : {df_clean[TARGET_COL].dtype}")
print(f"Unique values: {sorted(df_clean[TARGET_COL].dropna().unique().tolist())}")
print(f"Missing      : {df_clean[TARGET_COL].isna().sum()}")

# Check for invalid labels
valid_labels = {0, 1}
actual_labels = set(df_clean[TARGET_COL].dropna().unique())
invalid_labels = actual_labels - valid_labels

if invalid_labels:
    print(f"\n⚠️  Invalid target labels found: {invalid_labels}")
else:
    print(f"\n✅ Target labels are valid: {sorted(actual_labels)}")

Target column: Fraud_Label
Dtype        : int64
Unique values: [0, 1]
Missing      : 0

✅ Target labels are valid: [np.int64(0), np.int64(1)]


In [18]:
# Target distribution
target_counts = df_clean[TARGET_COL].value_counts().sort_index()
target_pcts = df_clean[TARGET_COL].value_counts(normalize=True).sort_index() * 100

target_summary = pd.DataFrame({
    "Count": target_counts,
    "Percentage (%)": target_pcts.round(2),
})
target_summary.index.name = TARGET_COL

print("Target Variable Distribution:")
display(target_summary)

fraud_pct = target_pcts.get(1, 0)
print(f"\nNote: The synthetic dataset has a fraud rate of {fraud_pct:.1f}%, which is")
print("considerably higher than real-world fraud rates. No class rebalancing is")
print("performed in this cleaning notebook — that decision belongs to later stages.")

Target Variable Distribution:


,Count,Percentage (%)
Fraud_Label,,
0,33933,67.87
1,16067,32.13



Note: The synthetic dataset has a fraud rate of 32.1%, which is
considerably higher than real-world fraud rates. No class rebalancing is
performed in this cleaning notebook — that decision belongs to later stages.


---
## 13. Identifier Handling Decision

Document how identifier columns are handled in this cleaning notebook.

In [19]:
IDENTIFIER_COLUMNS = ["Transaction_ID", "User_ID"]

print("IDENTIFIER COLUMN HANDLING DECISION")
print("=" * 55)

for col in IDENTIFIER_COLUMNS:
    n_unique = df_clean[col].nunique()
    n_total = len(df_clean)
    unique_ratio = n_unique / n_total
    
    print(f"\n  {col}:")
    print(f"    Unique values: {n_unique:,} / {n_total:,} ({unique_ratio:.2%})")
    print(f"    Sample values: {df_clean[col].head(5).tolist()}")

print("\n" + "-" * 55)
print("\n  DECISIONS:")
print("  • Transaction_ID — unique transaction identifier.")
print("    → Retained in the cleaned dataset.")
print("    → Must NOT be used as a predictive ML feature.")
print()
print("  • User_ID — customer/user identifier.")
print("    → Retained in the cleaned dataset.")
print("    → Must NOT be directly used as a predictive ML feature.")
print("    → MUST remain available for later customer-level")
print("      behavioral feature engineering (e.g., aggregations,")
print("      transaction frequency per user, etc.).")
print()
print("  IMPORTANT: High cardinality alone does NOT make a column")
print("  an identifier. Columns like Account_Balance have many")
print("  unique values but are legitimate numerical features.")
print("  Only Transaction_ID and User_ID are true identifiers.")

IDENTIFIER COLUMN HANDLING DECISION

  Transaction_ID:
    Unique values: 50,000 / 50,000 (100.00%)
    Sample values: ['TXN_33553', 'TXN_9427', 'TXN_199', 'TXN_12447', 'TXN_39489']

  User_ID:
    Unique values: 8,963 / 50,000 (17.93%)
    Sample values: ['USER_1834', 'USER_7875', 'USER_2734', 'USER_2617', 'USER_2014']

-------------------------------------------------------

  DECISIONS:
  • Transaction_ID — unique transaction identifier.
    → Retained in the cleaned dataset.
    → Must NOT be used as a predictive ML feature.

  • User_ID — customer/user identifier.
    → Retained in the cleaned dataset.
    → Must NOT be directly used as a predictive ML feature.
    → MUST remain available for later customer-level
      behavioral feature engineering (e.g., aggregations,
      transaction frequency per user, etc.).

  IMPORTANT: High cardinality alone does NOT make a column
  an identifier. Columns like Account_Balance have many
  unique values but are legitimate numerical features

---
## 14. Cleaning Operations

Apply only justified cleaning operations based on the validations above.

**Operations performed in this notebook:**
1. Convert `Date` to `datetime` (already done in Section 9)
2. Correct numeric dtypes (already done in Section 6)
3. Remove exact duplicate rows if they exist (handled in Section 8)
4. Handle genuinely invalid/missing values only if they exist
5. Standardize blank categorical strings if necessary

**NOT performed:**
- Scaling / normalisation
- Encoding categorical variables
- Feature engineering
- Class rebalancing
- Outlier removal
- Dropping `User_ID` or `Transaction_ID`

In [20]:
# Track cleaning actions
cleaning_log = []

# 1. Date conversion — already performed in Section 9
cleaning_log.append("Date column converted from text to datetime (Section 9).")

# 2. Numeric dtype corrections — already performed in Section 6
cleaning_log.append("Numeric columns validated and converted safely (Section 6).")

# 3. Duplicate removal — already handled in Section 8
cleaning_log.append("Duplicate check performed (Section 8).")

# 4. Standardize blank/whitespace categorical strings
blank_fixed = 0
for col in CATEGORICAL_COLUMNS:
    # Replace blank/whitespace-only strings with NaN
    mask = df_clean[col].astype(str).str.strip().eq("")
    n_blank = mask.sum()
    if n_blank > 0:
        df_clean.loc[mask, col] = np.nan
        blank_fixed += n_blank
        cleaning_log.append(f"{col}: {n_blank} blank string(s) replaced with NaN.")

    # Strip leading/trailing whitespace from valid strings
    df_clean[col] = df_clean[col].astype(str).str.strip()
    # Restore NaN where 'nan' string was created
    df_clean.loc[df_clean[col] == "nan", col] = np.nan

if blank_fixed == 0:
    cleaning_log.append("No blank categorical strings found — no standardization needed.")

# 5. Identifier columns ensured as strings — already in Section 6
cleaning_log.append("Identifier columns (Transaction_ID, User_ID) confirmed as string type.")

print("Cleaning operations applied:\n")
for i, action in enumerate(cleaning_log, 1):
    print(f"  {i}. {action}")

Cleaning operations applied:

  1. Date column converted from text to datetime (Section 9).
  2. Numeric columns validated and converted safely (Section 6).
  3. Duplicate check performed (Section 8).
  4. No blank categorical strings found — no standardization needed.
  5. Identifier columns (Transaction_ID, User_ID) confirmed as string type.


---
## 15. Before vs After Validation

Compare the dataset before and after cleaning to clearly demonstrate what changed.

In [21]:
comparison = pd.DataFrame({
    "Metric": [
        "Rows",
        "Columns",
        "Total missing values",
        "Duplicate rows",
        "Date column dtype",
        "Invalid target values",
    ],
    "Before (Raw)": [
        f"{df_raw.shape[0]:,}",
        f"{df_raw.shape[1]}",
        f"{df_raw.isnull().sum().sum():,}",
        f"{df_raw.duplicated().sum():,}",
        str(df_raw["Date"].dtype),
        str((~df_raw["Fraud_Label"].isin([0, 1])).sum()),
    ],
    "After (Cleaned)": [
        f"{df_clean.shape[0]:,}",
        f"{df_clean.shape[1]}",
        f"{df_clean.isnull().sum().sum():,}",
        f"{df_clean.duplicated().sum():,}",
        str(df_clean["Date"].dtype),
        str((~df_clean["Fraud_Label"].isin([0, 1])).sum()),
    ],
})

comparison = comparison.set_index("Metric")
print("Before vs After Cleaning Comparison:")
display(comparison)

Before vs After Cleaning Comparison:


,Before (Raw),After (Cleaned)
Metric,,
Rows,"50,000","50,000"
Columns,14,14
Total missing values,0,0
Duplicate rows,0,0
Date column dtype,str,datetime64[us]
Invalid target values,0,0


---
## 16. Save Cleaned Dataset

In [22]:
# Save the cleaned dataset
df_clean.to_csv(PROCESSED_DATA_PATH, index=False)

print(f"✅ Cleaned dataset saved to: {PROCESSED_DATA_PATH}")
print(f"   File exists: {PROCESSED_DATA_PATH.exists()}")
print(f"   File size  : {PROCESSED_DATA_PATH.stat().st_size:,} bytes")

✅ Cleaned dataset saved to: C:\Users\sarva\Desktop\financial-fraud-detection\data\processed\cleaned_fraud_dataset.csv
   File exists: True
   File size  : 4,916,963 bytes


In [23]:
# Verify by reloading the saved file
df_reload = pd.read_csv(PROCESSED_DATA_PATH)

print("Reload verification:")
print(f"  Shape         : {df_reload.shape[0]:,} rows × {df_reload.shape[1]} columns")
print(f"  Columns       : {list(df_reload.columns)}")
print(f"  Date dtype    : {df_reload['Date'].dtype}")
print(f"  Target values : {sorted(df_reload['Fraud_Label'].unique().tolist())}")

# Note on CSV datetime behaviour
print("\n  Note: CSV does not preserve pandas datetime dtype. After reloading,")
print("  the Date column is stored as text (object). Downstream notebooks must")
print("  parse it back to datetime with: pd.to_datetime(df['Date'])")

# Verify shape matches
assert df_reload.shape == df_clean.shape, (
    f"Shape mismatch: saved {df_clean.shape}, reloaded {df_reload.shape}"
)
print("\n✅ Reload verification passed — shapes match.")

Reload verification:
  Shape         : 50,000 rows × 14 columns
  Columns       : ['Transaction_ID', 'User_ID', 'Transaction_Amount', 'Transaction_Type', 'Date', 'Account_Balance', 'Device_Type', 'Location', 'Merchant_Category', 'Previous_Fraudulent_Activity', 'Daily_Transaction_Count', 'Card_Type', 'Card_Age', 'Fraud_Label']
  Date dtype    : str
  Target values : [0, 1]

  Note: CSV does not preserve pandas datetime dtype. After reloading,
  the Date column is stored as text (object). Downstream notebooks must
  parse it back to datetime with: pd.to_datetime(df['Date'])

✅ Reload verification passed — shapes match.


---
## 17. Final Data Quality Summary

In [24]:
# Build the final quality summary dynamically
n_missing_final = df_clean.isnull().sum().sum()
n_dup_final = df_clean.duplicated().sum()
date_parsed_ok = df_clean["Date"].dtype == "datetime64[ns]"
n_unparsed_dates = df_clean["Date"].isna().sum()
invalid_target = (~df_clean["Fraud_Label"].isin([0, 1])).sum()

quality_checks = pd.DataFrame([
    {
        "Check": "Required columns present",
        "Status": "✅ PASS",
        "Details": f"All {len(EXPECTED_COLUMNS)} columns found.",
    },
    {
        "Check": "Missing values",
        "Status": "✅ PASS" if n_missing_final == 0 else "⚠️ WARN",
        "Details": f"{n_missing_final} missing values remaining.",
    },
    {
        "Check": "Duplicate rows",
        "Status": "✅ PASS" if n_dup_final == 0 else "⚠️ WARN",
        "Details": f"{n_dup_final} duplicates remaining.",
    },
    {
        "Check": "Date parsing",
        "Status": "✅ PASS" if date_parsed_ok and n_unparsed_dates == 0 else "⚠️ WARN",
        "Details": f"Dtype: {df_clean['Date'].dtype}, {n_unparsed_dates} unparsed.",
    },
    {
        "Check": "Invalid numeric values",
        "Status": "✅ PASS" if n_missing_final == 0 else "⚠️ REVIEW",
        "Details": "No invalid values introduced during conversion." if n_missing_final == 0 else "Check Section 11.",
    },
    {
        "Check": "Target validity",
        "Status": "✅ PASS" if invalid_target == 0 else "❌ FAIL",
        "Details": f"Values: {sorted(df_clean['Fraud_Label'].dropna().unique().tolist())}, {invalid_target} invalid.",
    },
    {
        "Check": "Identifier handling",
        "Status": "✅ PASS",
        "Details": "Transaction_ID and User_ID retained; documented as non-features.",
    },
    {
        "Check": "Raw dataset preserved",
        "Status": "✅ PASS" if RAW_DATA_PATH.exists() else "❌ FAIL",
        "Details": f"{RAW_DATA_PATH.name} exists and was not modified.",
    },
    {
        "Check": "Processed dataset saved",
        "Status": "✅ PASS" if PROCESSED_DATA_PATH.exists() else "❌ FAIL",
        "Details": f"{PROCESSED_DATA_PATH.name} saved successfully.",
    },
])

quality_checks = quality_checks.set_index("Check")
display(quality_checks)

,Status,Details
Check,,
Required columns present,✅ PASS,All 14 columns found.
Missing values,✅ PASS,0 missing values remaining.
Duplicate rows,✅ PASS,0 duplicates remaining.
Date parsing,⚠️ WARN,"Dtype: datetime64[us], 0 unparsed."
Invalid numeric values,✅ PASS,No invalid values introduced during conversion.
Target validity,✅ PASS,"Values: [0, 1], 0 invalid."
Identifier handling,✅ PASS,Transaction_ID and User_ID retained; documente...
Raw dataset preserved,✅ PASS,synthetic_fraud_dataset.csv exists and was not...
Processed dataset saved,✅ PASS,cleaned_fraud_dataset.csv saved successfully.


In [25]:
print("=" * 60)
print("FINAL SUMMARY")
print("=" * 60)
print(f"")
print(f"  Dataset rows (raw)     : {df_raw.shape[0]:,}")
print(f"  Dataset rows (cleaned) : {df_clean.shape[0]:,}")
print(f"  Rows removed           : {df_raw.shape[0] - df_clean.shape[0]:,}")
print(f"  Columns                : {df_clean.shape[1]}")
print(f"  Missing values         : {n_missing_final}")
print(f"  Duplicate rows         : {n_dup_final}")
print(f"  Date column dtype      : {df_clean['Date'].dtype}")
print(f"  Target values          : {sorted(df_clean['Fraud_Label'].dropna().unique().tolist())}")
print(f"  Identifiers retained   : Transaction_ID, User_ID")
print(f"  Raw file preserved     : {RAW_DATA_PATH.exists()}")
print(f"  Cleaned file saved     : {PROCESSED_DATA_PATH.exists()}")
print(f"")
print("The raw dataset has been validated, cleaned, and saved.")
print("No model performance is discussed because no model exists yet.")

FINAL SUMMARY

  Dataset rows (raw)     : 50,000
  Dataset rows (cleaned) : 50,000
  Rows removed           : 0
  Columns                : 14
  Missing values         : 0
  Duplicate rows         : 0
  Date column dtype      : datetime64[us]
  Target values          : [0, 1]
  Identifiers retained   : Transaction_ID, User_ID
  Raw file preserved     : True
  Cleaned file saved     : True

The raw dataset has been validated, cleaned, and saved.
No model performance is discussed because no model exists yet.


---
## 18. Next Steps

**Notebook 02 completed: the dataset has been validated and cleaned.**

The raw dataset has been transformed into a clean, reliable dataset saved at `data/processed/cleaned_fraud_dataset.csv`.

The next stage will be:

> **Notebook 03 — Exploratory Data Analysis (EDA)**
>
> This notebook will perform in-depth exploration of the cleaned dataset, including distribution analysis, feature-target relationships, correlation analysis, and visual insights to inform later feature engineering and modelling decisions.

---

**Next notebook: `03_eda.ipynb`**